# Electrolyte Imbalance Model Rebuild

Compact, VS-Code-friendly notebook script for predicting `electrolyte_imbalance`
from a short quiz plus optional demographics / medication count / standard lab upload.

Key goals:
- keep feature sets product-friendly
- include `age`, `gender`, `med_count`
- include standard screening/check-up style labs where available
- avoid direct leakage from the electrolyte target definition
- use scaled + tuned logistic regression
- compare compact runs side by side
- support threshold selection and model export

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import re
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.inspection import permutation_importance

In [2]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_electrolyte_imbalance_rebuild")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "electrolyte_imbalance"
RANDOM_STATE = 42
TEST_SIZE = 0.20

MED_COUNT_CANDIDATES = ["med_count", "medication_count", "n_medications", "rx_count"]

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

DATA_PATH: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild


In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
print("Target present:", TARGET in df.columns)
assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print("\nTarget distribution:")
print(df[TARGET].value_counts(dropna=False).sort_index())
print("\nTarget prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

Shape: (7437, 878)
Target present: True

Target distribution:
electrolyte_imbalance
0    6858
1     579
Name: count, dtype: int64

Target prevalence:
electrolyte_imbalance
0    0.9221
1    0.0779
Name: proportion, dtype: float64


## Basic cleanup
Replace common NHANES-style special response codes (7/9/77/99 etc.) with NaN
in questionnaire-like columns.

In [4]:
SPECIAL_MISSING_CODES = {
    7, 9,
    77, 99,
    777, 999,
    7777, 9999,
    77777, 99999,
}

QUESTIONNAIRE_LIKE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq",
    "alq", "whq", "rhq", "hsq", "ocq", "pfq"
)

def replace_special_codes_with_nan(series: pd.Series) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(series):
        return series
    return series.replace(list(SPECIAL_MISSING_CODES), np.nan)

questionnaire_like_cols = [
    c for c in df.columns
    if c.lower().startswith(QUESTIONNAIRE_LIKE_PREFIXES)
]

df[questionnaire_like_cols] = df[questionnaire_like_cols].apply(replace_special_codes_with_nan)

print("Cleaned questionnaire-like columns:", len(questionnaire_like_cols))

Cleaned questionnaire-like columns: 162


## Medication count
Reuse an existing medication-count column if available, otherwise derive it from
`medication_*` indicator columns.

In [5]:
med_count_col = None

for c in MED_COUNT_CANDIDATES:
    if c in df.columns:
        med_count_col = c
        break

if med_count_col is None:
    medication_cols = [c for c in df.columns if c.lower().startswith("medication_")]
    if medication_cols:
        df["med_count"] = (
            df[medication_cols]
            .notna()
            .sum(axis=1)
            .astype(float)
        )
        med_count_col = "med_count"

if med_count_col and med_count_col != "med_count":
    df["med_count"] = df[med_count_col].copy()
    med_count_col = "med_count"

print("Medication count column:", med_count_col)

Medication count column: med_count


## Optional alcohol risk feature engineering
Electrolyte imbalance can be influenced by hydration / alcohol patterns.
We derive one compact alcohol risk flag if enough raw columns are available.

In [6]:
ALCOHOL_RAW_COLS = [
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq121___past_12_mo_how_often_drink_alcoholic_bev",
    "alq130___avg_#_alcoholic_drinks/day___past_12_mos",
    "alq142___#_days_have_4_or_5_drinks/past_12_mos",
    "alq151___ever_have_4/5_or_more_drinks_every_day?",
    "alq170___past_30_days_#_times_4_5_drinks_on_an_oc",
]
available_alcohol_cols = [c for c in ALCOHOL_RAW_COLS if c in df.columns]
print("Available alcohol raw cols:", available_alcohol_cols)

Available alcohol raw cols: ['alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc']


In [7]:
if "alq111___ever_had_a_drink_of_any_kind_of_alcohol" in df.columns:
    s = df["alq111___ever_had_a_drink_of_any_kind_of_alcohol"]
    df["alcohol_ever_yes"] = np.where(s.isna(), np.nan, (s == 1).astype(float))

if "alq142___#_days_have_4_or_5_drinks/past_12_mos" in df.columns:
    s = pd.to_numeric(df["alq142___#_days_have_4_or_5_drinks/past_12_mos"], errors="coerce")
    df["alcohol_binge_any_12m"] = np.where(s.isna(), np.nan, (s > 0).astype(float))
    df["alcohol_binge_monthly_or_more"] = np.where(s.isna(), np.nan, (s >= 12).astype(float))

if "alq170___past_30_days_#_times_4_5_drinks_on_an_oc" in df.columns:
    s = pd.to_numeric(df["alq170___past_30_days_#_times_4_5_drinks_on_an_oc"], errors="coerce")
    df["alcohol_binge_any_30d"] = np.where(s.isna(), np.nan, (s > 0).astype(float))

if "alq151___ever_have_4/5_or_more_drinks_every_day?" in df.columns:
    s = df["alq151___ever_have_4/5_or_more_drinks_every_day?"]
    df["alcohol_ever_daily_heavy"] = np.where(s.isna(), np.nan, (s == 1).astype(float))

alcohol_risk_inputs = [
    c for c in [
        "alcohol_binge_any_12m",
        "alcohol_binge_any_30d",
        "alcohol_binge_monthly_or_more",
        "alcohol_ever_daily_heavy",
    ]
    if c in df.columns
]

if alcohol_risk_inputs:
    df["alcohol_any_risk_signal"] = df[alcohol_risk_inputs].max(axis=1, skipna=True)
    if df[alcohol_risk_inputs].isna().all(axis=1).any():
        df.loc[df[alcohol_risk_inputs].isna().all(axis=1), "alcohol_any_risk_signal"] = np.nan

print("Derived alcohol features:", [c for c in ["alcohol_any_risk_signal"] if c in df.columns])

Derived alcohol features: ['alcohol_any_risk_signal']


## Candidate feature groups
Compact, product-oriented feature sets.  
No direct electrolyte leakage features.

In [8]:
DEMO_COLS = [
    "age_years",
    "gender",
]

MED_COLS = [
    "med_count",
]

# Short quiz features that are at least somewhat plausible for a fatigue app
# and were already explored in the old EI notebook.
EI_QUIZ_COLS = [
    "bpq020___ever_told_you_had_high_blood_pressure",
    "kiq480___how_many_times_urinate_in_night?",
    "paq650___vigorous_recreational_activities",
    "dpq040___feeling_tired_or_having_little_energy",
    "kiq026___ever_had_kidney_stones?",
    "kiq022___ever_told_you_had_weak/failing_kidneys?",
    "mcq160a___ever_told_you_had_arthritis",
    "kiq005___how_often_have_urinary_leakage?",
    "alcohol_any_risk_signal",
]

# Standard check-up / common screening style labs that may be available from uploads.
# These are not classic electrolyte labs, but they are realistic upload candidates and
# can provide risk context without direct target leakage.
SCREENING_LAB_COLS = [
    "total_cholesterol_mg_dl",
    "hdl_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "uacr_mg_g",
]

# Optional augmented labs for research. These may improve signal, but are less standard
# for a broad public health check-up flow. Avoid direct sodium/potassium/calcium leakage.
AUGMENTED_CONTEXT_LAB_COLS = [
    "serum_creatinine_mg_dl",
    "uacr_mg_g",
    "blood_urea_nitrogen_mg_dl",
    "uric_acid_mg_dl",
]

LEAKAGE_COLS = [
    TARGET,
    "LBXSNASI_sodium_mmol_l",
    "LBXSKSI_potassium_mmol_l",
    "LBXSCA_total_calcium_mg_dl",
    "sodium_mmol_l",
    "potassium_mmol_l",
    "chloride_mmol_l",
    "bicarbonate_mmol_l",
    "co2_mmol_l",
]

In [9]:
def existing_cols(cols, frame):
    return [c for c in cols if c in frame.columns and c not in LEAKAGE_COLS]

def dedupe_keep_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

feature_sets = {
    "compact_quiz_only": dedupe_keep_order(
        existing_cols(EI_QUIZ_COLS, df)
    ),
    "compact_quiz_demo_med": dedupe_keep_order(
        existing_cols(EI_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
    ),
    "compact_quiz_demo_med_screening_labs": dedupe_keep_order(
        existing_cols(EI_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
    ),
    "compact_quiz_demo_med_augmented_labs": dedupe_keep_order(
        existing_cols(EI_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
        + existing_cols(AUGMENTED_CONTEXT_LAB_COLS, df)
    ),
}

for run_name, feats in feature_sets.items():
    print(f"{run_name}: {len(feats)} features")
    print(feats)
    print()

compact_quiz_only: 9 features
['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'dpq040___feeling_tired_or_having_little_energy', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'mcq160a___ever_told_you_had_arthritis', 'kiq005___how_often_have_urinary_leakage?', 'alcohol_any_risk_signal']

compact_quiz_demo_med: 12 features
['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'dpq040___feeling_tired_or_having_little_energy', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'mcq160a___ever_told_you_had_arthritis', 'kiq005___how_often_have_urinary_leakage?', 'alcohol_any_risk_signal', 'age_years', 'gender', 'med_count']

compact_quiz_demo_med_screening_labs: 16 features
['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_

## Shared helper functions

In [10]:
def is_binary_series(s: pd.Series) -> bool:
    vals = set(pd.Series(s.dropna()).unique())
    return vals.issubset({0, 1}) and len(vals) <= 2

def infer_feature_type(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        unique_non_null = s.dropna().nunique()
        if is_binary_series(s):
            return "categorical"
        if unique_non_null <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

def build_xy(frame: pd.DataFrame, target: str, feature_list):
    model_df = frame[[target] + feature_list].copy()
    mask = model_df[target].notna()
    X = model_df.loc[mask, feature_list].copy()
    y = model_df.loc[mask, target].astype(int).copy()
    return X, y

def get_feature_types(X_train: pd.DataFrame, feature_list):
    numeric_features = []
    categorical_features = []
    for col in feature_list:
        if infer_feature_type(X_train[col]) == "numeric":
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    return numeric_features, categorical_features

def evaluate_predictions(y_true, y_proba, threshold=0.50):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "avg_precision": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }

numeric_transformer_logreg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_transformer_logreg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

def build_logreg_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_logreg, numeric_features),
            ("cat", categorical_transformer_logreg, categorical_features),
        ],
        remainder="drop",
    )

def run_tuned_logreg(
    X_train,
    X_test,
    y_train,
    y_test,
    numeric_features,
    categorical_features,
    model_name="logreg_tuned",
):
    preprocessor = build_logreg_preprocessor(numeric_features, categorical_features)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ))
    ])

    param_grid = [
        {
            "model__solver": ["liblinear"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
        {
            "model__solver": ["saga"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
    ]

    cv = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=2,
        random_state=RANDOM_STATE,
    )

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    metrics = evaluate_predictions(y_test, y_proba, threshold=0.50)

    result = {
        "model_name": model_name,
        "best_params": grid.best_params_,
        "cv_best_score_avg_precision": grid.best_score_,
        "roc_auc": metrics["roc_auc"],
        "avg_precision": metrics["avg_precision"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
        "best_estimator": best_model,
        "grid_search": grid,
        "y_test_true": y_test.copy(),
        "y_test_proba": y_proba.copy(),
    }

    return result

def prettify_transformed_feature_name(name: str) -> str:
    FEATURE_LABEL_MAP = {
        "age_years": "Age (years)",
        "gender": "Gender",
        "med_count": "Medication count",
        "bpq020___ever_told_you_had_high_blood_pressure": "Ever told had high blood pressure",
        "kiq480___how_many_times_urinate_in_night?": "Urinate at night",
        "paq650___vigorous_recreational_activities": "Vigorous recreational activities",
        "dpq040___feeling_tired_or_having_little_energy": "Tired / little energy",
        "kiq026___ever_had_kidney_stones?": "Ever had kidney stones",
        "kiq022___ever_told_you_had_weak/failing_kidneys?": "Ever told had weak/failing kidneys",
        "mcq160a___ever_told_you_had_arthritis": "Ever told had arthritis",
        "kiq005___how_often_have_urinary_leakage?": "How often urinary leakage",
        "alcohol_any_risk_signal": "Alcohol risk signal",
        "total_cholesterol_mg_dl": "Total cholesterol",
        "hdl_cholesterol_mg_dl": "HDL cholesterol",
        "triglycerides_mg_dl": "Triglycerides",
        "uacr_mg_g": "UACR",
        "serum_creatinine_mg_dl": "Serum creatinine",
        "blood_urea_nitrogen_mg_dl": "Blood urea nitrogen",
        "uric_acid_mg_dl": "Uric acid",
    }

    VALUE_LABEL_MAP = {
        "gender": {
            "Male": "Male",
            "Female": "Female",
        },
        "bpq020___ever_told_you_had_high_blood_pressure": {
            "1.0": "Yes",
            "2.0": "No",
        },
        "paq650___vigorous_recreational_activities": {
            "1.0": "Yes",
            "2.0": "No",
        },
        "kiq026___ever_had_kidney_stones?": {
            "1.0": "Yes",
            "2.0": "No",
        },
        "kiq022___ever_told_you_had_weak/failing_kidneys?": {
            "1.0": "Yes",
            "2.0": "No",
        },
        "mcq160a___ever_told_you_had_arthritis": {
            "1.0": "Yes",
            "2.0": "No",
        },
        "alcohol_any_risk_signal": {
            "0.0": "No risk signal",
            "1.0": "Any risk signal",
        },
    }

    if name.startswith("num__missingindicator_"):
        raw = name.replace("num__missingindicator_", "")
        pretty_raw = FEATURE_LABEL_MAP.get(raw, raw)
        return f"{pretty_raw} missing"

    if name.startswith("num__"):
        raw = name.replace("num__", "")
        return FEATURE_LABEL_MAP.get(raw, raw)

    if name.startswith("cat__"):
        raw = name.replace("cat__", "")
        if "_" in raw:
            feature_part, value_part = raw.rsplit("_", 1)
            pretty_feature = FEATURE_LABEL_MAP.get(feature_part, feature_part)
            pretty_value = VALUE_LABEL_MAP.get(feature_part, {}).get(value_part, value_part)
            return f"{pretty_feature}: {pretty_value}"
        return FEATURE_LABEL_MAP.get(raw, raw)

    return name

def extract_pretty_coefficients(selected_result):
    best_estimator = selected_result["best_estimator"]
    preprocessor = best_estimator.named_steps["preprocessor"]
    model = best_estimator.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()

    coef_df_raw = pd.DataFrame({
        "transformed_feature": feature_names,
        "coefficient": model.coef_[0],
    })
    coef_df_raw["abs_coefficient"] = coef_df_raw["coefficient"].abs()
    coef_df_raw = coef_df_raw.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

    coef_df_pretty = coef_df_raw.copy()
    coef_df_pretty["pretty_feature"] = coef_df_pretty["transformed_feature"].apply(prettify_transformed_feature_name)
    coef_df_pretty = coef_df_pretty[
        ["pretty_feature", "coefficient", "abs_coefficient", "transformed_feature"]
    ]

    return coef_df_pretty

## Shared holdout split
Use one shared split across all runs for fair comparison.

In [11]:
shared_features = dedupe_keep_order(
    [f for feats in feature_sets.values() for f in feats]
)

X_all, y_all = build_xy(df, TARGET, shared_features)

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

print("Shared train shape:", X_train_all.shape)
print("Shared test shape:", X_test_all.shape)
print("Train prevalence:", y_train.mean().round(4))
print("Test prevalence:", y_test.mean().round(4))

Shared train shape: (5949, 17)
Shared test shape: (1488, 17)
Train prevalence: 0.0778
Test prevalence: 0.078


## Run compact model comparisons

In [12]:
all_results = {}
comparison_rows = []

for run_name, feature_list in feature_sets.items():
    X_train_run = X_train_all[feature_list].copy()
    X_test_run = X_test_all[feature_list].copy()

    numeric_features, categorical_features = get_feature_types(X_train_run, feature_list)

    baseline_model = DummyClassifier(strategy="most_frequent")
    baseline_model.fit(X_train_run, y_train)
    baseline_pred = baseline_model.predict(X_test_run)

    result = run_tuned_logreg(
        X_train=X_train_run,
        X_test=X_test_run,
        y_train=y_train,
        y_test=y_test,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        model_name=run_name,
    )

    result["feature_list"] = feature_list
    result["numeric_features"] = numeric_features
    result["categorical_features"] = categorical_features
    result["baseline_precision"] = precision_score(y_test, baseline_pred, zero_division=0)
    result["baseline_recall"] = recall_score(y_test, baseline_pred, zero_division=0)
    result["baseline_f1"] = f1_score(y_test, baseline_pred, zero_division=0)

    all_results[run_name] = result

    comparison_rows.append({
        "run": run_name,
        "n_features": len(feature_list),
        "cv_best_score_avg_precision": round(result["cv_best_score_avg_precision"], 4),
        "roc_auc": round(result["roc_auc"], 4),
        "avg_precision": round(result["avg_precision"], 4),
        "precision": round(result["precision"], 4),
        "recall": round(result["recall"], 4),
        "f1": round(result["f1"], 4),
        "best_params": json.dumps(result["best_params"]),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["avg_precision", "roc_auc", "recall"],
    ascending=[False, False, False],
).reset_index(drop=True)

comparison_df

Fitting 10 folds for each of 20 candidates, totalling 200 fits
Fitting 10 folds for each of 20 candidates, totalling 200 fits
Fitting 10 folds for each of 20 candidates, totalling 200 fits
Fitting 10 folds for each of 20 candidates, totalling 200 fits


,run,n_features,cv_best_score_avg_precision,roc_auc,avg_precision,precision,recall,f1,best_params
0,compact_quiz_demo_med_augmented_labs,17,0.1856,0.7243,0.1880,0.1426,0.6810,0.2358,"{""model__C"": 0.01, ""model__penalty"": ""l2"", ""mo..."
1,compact_quiz_demo_med_screening_labs,16,0.1846,0.7219,0.1875,0.1392,0.6638,0.2302,"{""model__C"": 0.01, ""model__penalty"": ""l2"", ""mo..."
2,compact_quiz_only,9,0.1395,0.6598,0.1616,0.1366,0.6207,0.2240,"{""model__C"": 0.01, ""model__penalty"": ""l2"", ""mo..."
3,compact_quiz_demo_med,12,0.1483,0.6770,0.1564,0.1469,0.6207,0.2376,"{""model__C"": 0.01, ""model__penalty"": ""l2"", ""mo..."


Prefer the compact screening-labs run as a product candidate, even if another run
is slightly stronger on raw metrics.

In [13]:
auto_best_run = comparison_df.iloc[0]["run"]
auto_best_result = all_results[auto_best_run]

selected_run = (
    "compact_quiz_demo_med_screening_labs"
    if "compact_quiz_demo_med_screening_labs" in all_results
    else auto_best_run
)
selected_result = all_results[selected_run]

print("Auto-best run:", auto_best_run)
print("Selected run:", selected_run)
print("Selected params:", selected_result["best_params"])
print("ROC-AUC:", round(selected_result["roc_auc"], 4))
print("Average Precision:", round(selected_result["avg_precision"], 4))
print("Precision:", round(selected_result["precision"], 4))
print("Recall:", round(selected_result["recall"], 4))
print("F1:", round(selected_result["f1"], 4))
print()
print("Confusion Matrix:")
print(selected_result["confusion_matrix"])
print()
print(selected_result["classification_report"])

Auto-best run: compact_quiz_demo_med_augmented_labs
Selected run: compact_quiz_demo_med_screening_labs
Selected params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
ROC-AUC: 0.7219
Average Precision: 0.1875
Precision: 0.1392
Recall: 0.6638
F1: 0.2302

Confusion Matrix:
[[896 476]
 [ 39  77]]

              precision    recall  f1-score   support

           0       0.96      0.65      0.78      1372
           1       0.14      0.66      0.23       116

    accuracy                           0.65      1488
   macro avg       0.55      0.66      0.50      1488
weighted avg       0.89      0.65      0.73      1488



## Threshold scan for the selected model

In [14]:
threshold_rows = []
y_true = selected_result["y_test_true"]
y_proba = selected_result["y_test_proba"]

roc_auc = roc_auc_score(y_true, y_proba)

for thr in np.arange(0.10, 0.91, 0.05):
    metrics = evaluate_predictions(y_true, y_proba, threshold=float(thr))
    threshold_rows.append({
        "threshold": round(float(thr), 2),
        "roc_auc": round(roc_auc, 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

,threshold,roc_auc,precision,recall,f1
0,0.10,0.7219,0.0908,1.0000,0.1664
1,0.15,0.7219,0.0910,1.0000,0.1668
2,0.20,0.7219,0.0911,1.0000,0.1669
3,0.25,0.7219,0.0913,1.0000,0.1673
4,0.30,0.7219,0.0914,0.9828,0.1673
5,0.35,0.7219,0.0957,0.9483,0.1739
6,0.40,0.7219,0.1057,0.8793,0.1887
7,0.45,0.7219,0.1127,0.7414,0.1957
8,0.50,0.7219,0.1392,0.6638,0.2302
9,0.55,0.7219,0.1785,0.5862,0.2736


## Coefficients of the selected model

In [15]:
coef_df_pretty = extract_pretty_coefficients(selected_result)
coef_df_pretty.head(40)

,pretty_feature,coefficient,abs_coefficient,transformed_feature
0,Total cholesterol missing,-0.558702,0.558702,num__missingindicator_total_cholesterol_mg_dl
1,HDL cholesterol missing,-0.558702,0.558702,num__missingindicator_hdl_cholesterol_mg_dl
2,Ever told had high blood pressure: No,-0.240773,0.240773,cat__bpq020___ever_told_you_had_high_blood_pre...
3,Ever told had high blood pressure: Yes,0.199750,0.199750,cat__bpq020___ever_told_you_had_high_blood_pre...
4,Urinate at night: 1.0,-0.182990,0.182990,cat__kiq480___how_many_times_urinate_in_night?...
5,Tired / little energy: 2.0,-0.179220,0.179220,cat__dpq040___feeling_tired_or_having_little_e...
6,Medication count,0.178154,0.178154,num__med_count
7,Vigorous recreational activities: Yes,-0.163063,0.163063,cat__paq650___vigorous_recreational_activities...
8,Urinate at night: 0.0,-0.156619,0.156619,cat__kiq480___how_many_times_urinate_in_night?...
9,Triglycerides missing,0.147939,0.147939,num__missingindicator_triglycerides_mg_dl


In [16]:
coef_df_nonzero = coef_df_pretty[coef_df_pretty["coefficient"] != 0].copy()
coef_df_nonzero = coef_df_nonzero.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
print("Non-zero coefficients:", len(coef_df_nonzero))
coef_df_nonzero

Non-zero coefficients: 39


,pretty_feature,coefficient,abs_coefficient,transformed_feature
0,Total cholesterol missing,-0.558702,0.558702,num__missingindicator_total_cholesterol_mg_dl
1,HDL cholesterol missing,-0.558702,0.558702,num__missingindicator_hdl_cholesterol_mg_dl
2,Ever told had high blood pressure: No,-0.240773,0.240773,cat__bpq020___ever_told_you_had_high_blood_pre...
3,Ever told had high blood pressure: Yes,0.199750,0.199750,cat__bpq020___ever_told_you_had_high_blood_pre...
4,Urinate at night: 1.0,-0.182990,0.182990,cat__kiq480___how_many_times_urinate_in_night?...
5,Tired / little energy: 2.0,-0.179220,0.179220,cat__dpq040___feeling_tired_or_having_little_e...
6,Medication count,0.178154,0.178154,num__med_count
7,Vigorous recreational activities: Yes,-0.163063,0.163063,cat__paq650___vigorous_recreational_activities...
8,Urinate at night: 0.0,-0.156619,0.156619,cat__kiq480___how_many_times_urinate_in_night?...
9,Triglycerides missing,0.147939,0.147939,num__missingindicator_triglycerides_mg_dl


## Permutation importance on original input features

In [17]:
perm = permutation_importance(
    selected_result["best_estimator"],
    X_test_all[feature_sets[selected_run]],
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": feature_sets[selected_run],
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

perm_df

,feature,importance_mean,importance_std
0,uacr_mg_g,0.016520,0.001948
1,bpq020___ever_told_you_had_high_blood_pressure,0.016258,0.006853
2,total_cholesterol_mg_dl,0.016068,0.008484
3,med_count,0.014362,0.012155
4,hdl_cholesterol_mg_dl,0.010451,0.010634
5,paq650___vigorous_recreational_activities,0.007820,0.004740
6,kiq480___how_many_times_urinate_in_night?,0.005701,0.004816
7,kiq026___ever_had_kidney_stones?,0.005678,0.001578
8,alcohol_any_risk_signal,0.005109,0.003508
9,gender,0.003277,0.002321


## Save artifacts

In [18]:
comparison_path = OUTPUT_DIR / "electrolyte_imbalance_run_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path.resolve())

coef_path = OUTPUT_DIR / f"{selected_run}_coefficients_pretty.csv"
coef_df_pretty.to_csv(coef_path, index=False)
print("Saved:", coef_path.resolve())

perm_path = OUTPUT_DIR / f"{selected_run}_permutation_importance.csv"
perm_df.to_csv(perm_path, index=False)
print("Saved:", perm_path.resolve())

threshold_path = OUTPUT_DIR / f"{selected_run}_threshold_scan.csv"
threshold_df.to_csv(threshold_path, index=False)
print("Saved:", threshold_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\electrolyte_imbalance_run_comparison.csv
Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\compact_quiz_demo_med_screening_labs_coefficients_pretty.csv
Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\compact_quiz_demo_med_screening_labs_permutation_importance.csv
Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\compact_quiz_demo_med_screening_labs_threshold_scan.csv


## Save selected model as joblib + metadata json

In [ ]:
disease = "electrolyte_imbalance"
threshold = 0.5
threshold_label = "05"

model_package = {
    "model": selected_result["best_estimator"],
    "threshold": threshold,
    "features": feature_sets[selected_run],
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
}

OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(exist_ok=True)


joblib_path = OUTPUT_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.joblib"
joblib.dump(model_package, joblib_path)

print("Joblib saved to:", joblib_path.resolve())

Joblib saved to: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\electrolyte_imbalance_compact_quiz_demo_med_screening_labs_threshold_05.joblib


In [20]:
metadata = {
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
    "threshold": threshold,
    "features": feature_sets[selected_run],
    "n_features": len(feature_sets[selected_run]),
    "metrics": {
        "roc_auc": float(selected_result["roc_auc"]),
        "avg_precision": float(selected_result["avg_precision"]),
        "precision": float(selected_result["precision"]),
        "recall": float(selected_result["recall"]),
        "f1": float(selected_result["f1"]),
    },
    "best_params": selected_result["best_params"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notes": (
        "Compact electrolyte imbalance model using quiz, demographics, medication count, "
        "and standard screening-style labs where available. Threshold 0.45 stored for a "
        "screening-oriented operating point."
    ),
}

metadata_path = OUTPUT_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadata saved to:", metadata_path.resolve())

Metadata saved to: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance_rebuild\electrolyte_imbalance_compact_quiz_demo_med_screening_labs_threshold_05.metadata.json


## Optional quick load test

In [21]:
loaded_package = joblib.load(joblib_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Loaded package keys:", loaded_package.keys())
print("Loaded threshold:", loaded_package["threshold"])
print("Loaded model name:", loaded_package["model_name"])
print("Loaded feature count:", len(loaded_package["features"]))
print("Loaded metadata model:", loaded_metadata["model_name"])

Loaded package keys: dict_keys(['model', 'threshold', 'features', 'model_name', 'target', 'disease'])
Loaded threshold: 0.5
Loaded model name: compact_quiz_demo_med_screening_labs
Loaded feature count: 16
Loaded metadata model: compact_quiz_demo_med_screening_labs


---
## RETRAIN v2 — Normalized Data Pipeline

### v2 Changes vs v1

| # | Change | v1 | v2 |
|---|--------|----|----||
| 1 | Data source | `nhanes_merged_adults_final.csv` (raw) | `nhanes_merged_adults_final_normalized.csv` (z-scored) |
| 2 | Scaling | `StandardScaler` in ColumnTransformer | Removed — data pre-normalized |
| 3 | Missingness | `SimpleImputer` no indicator | `SimpleImputer(median, add_indicator=True)` |
| 4 | Gender | raw string `gender` | binary `gender_female` (1=Female, 0=Male) |
| 5 | Model saving | dict-wrapped `.joblib` | plain pipeline `.joblib` |
| 6 | Evaluation | single train/test split | 5-fold stratified CV + OOF threshold sweep |
| 7 | Features | 16 roadmap features (predefined) | all 79 roadmap features → L1-select → deduplicate |
| 8 | Threshold | 0.50 (single) | pipeline_gate=0.35, recommended_threshold=0.60 |
| 9 | Alcohol feature | `alcohol_any_risk_signal` (derived, not in normalized file) | replaced by `alq111`, `alq151`, `alq130` |

### Leakage removed
- `LBXSNASI_sodium_mmol_l`, `LBXSKSI_potassium_mmol_l`, `LBXSCA_total_calcium_mg_dl` — direct electrolyte labs that define the target
- `sodium_mmol_l`, `potassium_mmol_l`, `chloride_mmol_l`, `bicarbonate_mmol_l`, `co2_mmol_l` — target components


In [ ]:
# ── v2 Setup ─────────────────────────────────────────────────────────────
# Run this cell first (no need to run any v1 cells above).
import os, warnings, json as _json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score)

warnings.filterwarnings("ignore")

DATA_V2       = "../data/processed/nhanes_merged_adults_final_normalized.csv"
MODEL_OUT_DIR = "../models_normalized"
TARGET_COL    = "electrolyte_imbalance"
RANDOM_STATE  = 42
CV_FOLDS      = 5

LEAKAGE_COLS = {
    TARGET_COL,
    "LBXSNASI_sodium_mmol_l", "LBXSKSI_potassium_mmol_l", "LBXSCA_total_calcium_mg_dl",
    "sodium_mmol_l", "potassium_mmol_l", "chloride_mmol_l", "bicarbonate_mmol_l", "co2_mmol_l",
}

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

# Read only the columns we actually need to avoid memory pressure
df = pd.read_csv(DATA_V2, low_memory=False)
df["gender_female"] = (df["gender"] == "Female").astype(float)
df["education_ord"] = df["education"].map({
    "Less than 9th grade":1, "9-11th grade":2, "High school diploma/GED":3,
    "Some college / AA":4, "College graduate or above":5})
df["pregnancy_status_bin"] = df["pregnancy_status"].map({"Yes, pregnant":1.0, "Not pregnant":0.0})

def build_pipe(C=1.0):
    return Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("clf", LogisticRegression(C=C, class_weight="balanced",
                                   max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])

def run_cv(feats, label, C=1.0):
    df_sub = df[feats + [TARGET_COL]].dropna(subset=[TARGET_COL])
    X = df_sub[feats]
    y = df_sub[TARGET_COL].astype(int)
    cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    prob = cross_val_predict(build_pipe(C), X, y, cv=cv, method="predict_proba")[:,1]
    auc = roc_auc_score(y, prob)
    p35 = precision_score(y, (prob>=0.35).astype(int), zero_division=0)
    r35 = recall_score(y, (prob>=0.35).astype(int), zero_division=0)
    print(f"{label}: AUC={auc:.4f}, n_feat={len(feats)}, P@0.35={p35:.3f}, R@0.35={r35:.3f}")
    return y, prob, auc

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]:,} cols")
print(f"Target distribution:")
print(df[TARGET_COL].value_counts(dropna=False))
print(f"Prevalence: {df[TARGET_COL].mean():.1%}")
print("Setup done.")


In [ ]:
# ── v2a  Load all roadmap features + L1 selection ────────────────────────
csv_path = '/Users/annaesakova/Downloads/HalfFull roadmap - diseases VS features (1).csv'
df_map = pd.read_csv(csv_path, header=1)

norm_cols = set(df.columns)
def find_col(row):
    for c in [str(row['canonical_feature']).strip(),
              str(row.get('mapped_dataset_column','')).strip()] + \
             [a.strip() for a in str(row.get('source_feature_names','')).split('|')]:
        if c in norm_cols: return c
    return None
df_map['norm_col'] = df_map.apply(find_col, axis=1)
df_map.loc[df_map['canonical_feature']=='gender','norm_col'] = 'gender_female'
df_map.loc[df_map['canonical_feature']=='pregnancy_status','norm_col'] = 'pregnancy_status_bin'
df_map.loc[df_map['canonical_feature']=='education','norm_col'] = 'education_ord'

usable = df_map[
    df_map['norm_col'].notna() &
    (df_map['feature_type'] != 'unresolved-alias') &
    ~df_map['norm_col'].isin(LEAKAGE_COLS)
].drop_duplicates('norm_col')
FEATURES_ALL = [f for f in usable['norm_col'].tolist() if df[f].dtype != object]
print(f'All roadmap features (no leakage): {len(FEATURES_ALL)}')

# L1 selection to find survivors
df_all = df[FEATURES_ALL + [TARGET_COL]].dropna(subset=[TARGET_COL])
X_all = df_all[FEATURES_ALL]
y_all = df_all[TARGET_COL].astype(int)

from sklearn.pipeline import Pipeline as _Pipe
pipe_l1 = _Pipe([
    ('imp', SimpleImputer(strategy='median', add_indicator=True)),
    ('clf', LogisticRegression(penalty='l1', C=0.1, class_weight='balanced',
                               max_iter=2000, solver='liblinear', random_state=RANDOM_STATE))
])
pipe_l1.fit(X_all, y_all)

imp_l1 = pipe_l1.named_steps['imp']
miss_names_l1 = [f'miss__{FEATURES_ALL[i]}' for i in imp_l1.indicator_.features_]
all_names_l1 = FEATURES_ALL + miss_names_l1
coefs_l1 = pipe_l1.named_steps['clf'].coef_[0]

L1_SELECTED = [all_names_l1[i] for i in range(len(coefs_l1))
               if coefs_l1[i] != 0 and not all_names_l1[i].startswith('miss__')]
# Always include age anchor
if 'age_years' not in L1_SELECTED: L1_SELECTED.append('age_years')

print(f'L1 survivors (non-zero coef): {len(L1_SELECTED)}')
for i, name in enumerate(all_names_l1):
    if name in L1_SELECTED:
        print(f'  {coefs_l1[i]:+.4f}  {name}')

y_l1, p_l1, auc_l1 = run_cv(L1_SELECTED, f'L1-{len(L1_SELECTED)}')


In [ ]:
# ── v2b  Spearman deduplication (|r| >= 0.45) ────────────────────────────
from scipy.stats import spearmanr
X_l1_filled = df[L1_SELECTED].fillna(df[L1_SELECTED].median())
corr_mat = X_l1_filled.corr(method='spearman')

print('Correlated pairs (|r| >= 0.45):')
for i in range(len(L1_SELECTED)):
    for j in range(i+1, len(L1_SELECTED)):
        r = corr_mat.iloc[i, j]
        if abs(r) >= 0.45:
            print(f'  r={r:+.3f}  {L1_SELECTED[i]}  <->  {L1_SELECTED[j]}')
print()
# Pairs found:
#   weight_kg <-> waist_cm (r=+0.813): keep weight_kg (stronger coef)
#   sld012 <-> sld013 (r=+0.451): keep sld012 (stronger coef)
# Also drop borderline features: rhq131 (|coef|=0.054), kiq450 (|coef|=0.052)
REMOVE_DUPS = {'waist_cm', 'sld013___sleep_hours___weekends', 'rhq131___ever_been_pregnant?',
               'kiq450___how_frequently_does_this_occur?'}
FEATURES_DEDUPED_28 = [f for f in L1_SELECTED if f not in REMOVE_DUPS]
print(f'After deduplication: {len(FEATURES_DEDUPED_28)} features')
for f in FEATURES_DEDUPED_28:
    print(f'  {f}')


In [ ]:
# ── v2c  Final feature list — 28 features ────────────────────────────────
# L1 on all 79 roadmap features → 58 survivors
# → remove weight_kg/waist_cm duplicate (r=0.813, keep weight_kg)
# → remove sld012/sld013 duplicate (r=0.451, keep sld012)
# → drop rhq131 (|coef|=0.054), kiq450 (|coef|=0.052)
#
# LEAKAGE EXCLUDED (target components — direct electrolyte labs):
#   sodium_mmol_l, potassium_mmol_l, chloride_mmol_l,
#   bicarbonate_mmol_l, co2_mmol_l, LBXSNASI_sodium_mmol_l,
#   LBXSKSI_potassium_mmol_l, LBXSCA_total_calcium_mg_dl

FEATURES_FINAL_28 = [
    # DEMOGRAPHICS
    'pregnancy_status_bin',             # strongest signal: pregnancy → electrolyte disruption
    'gender_female',                     # women higher risk
    'age_years',                         # clinical anchor
    # METABOLIC
    'fasting_glucose_mg_dl',             # osmotic diuresis → electrolyte loss
    'triglycerides_mg_dl',
    'hdl_cholesterol_mg_dl',
    'LBDLDL_ldl_cholesterol_friedewald_mg_dl',
    'LBXSTP_total_protein_g_dl',         # low protein → aldosterone dysregulation
    'LBXWBCSI_white_blood_cell_count_1000_cells_ul',  # immune/inflammatory signal
    # KIDNEY / URINARY
    'uacr_mg_g',                         # kidney damage marker
    'kiq022___ever_told_you_had_weak/failing_kidneys?',
    'kiq026___ever_had_kidney_stones?',
    'kiq042___leak_urine_during_physical_activities?',
    'kiq480___how_many_times_urinate_in_night?',
    # CARDIOVASCULAR / BP
    'bpq030___told_had_high_blood_pressure___2+_times',
    'bpq080___doctor_told_you___high_cholesterol_level',
    # SLEEP
    'slq050___ever_told_doctor_had_trouble_sleeping?',  # NOT EI-leakage; sleep → cortisol → electrolyte disruption
    'sld012___sleep_hours___weekdays_or_workdays',
    # ACTIVITY
    'paq650___vigorous_recreational_activities',  # vigorous exercise → electrolyte loss through sweat
    # MEDICATIONS
    'med_count',
    # ALCOHOL
    'alq151___ever_have_4/5_or_more_drinks_every_day?',  # heavy drinking → hypomagnesemia
    'alq130___avg_#_alcoholic_drinks/day___past_12_mos',
    # OTHER CONDITIONS
    'diq010___doctor_told_you_have_diabetes',
    'mcq160a___ever_told_you_had_arthritis',
    'cdq010___shortness_of_breath_on_stairs/inclines',
    # WEIGHT
    'weight_kg',
    # FEMALE REPRODUCTIVE
    'rhq540___ever_use_female_hormones?',
    'rhq060___age_at_last_menstrual_period',
]

assert len(FEATURES_FINAL_28) == 28, f'Expected 28, got {len(FEATURES_FINAL_28)}'
missing = [f for f in FEATURES_FINAL_28 if f not in df.columns]
leakage_check = [f for f in FEATURES_FINAL_28 if f in LEAKAGE_COLS]
print(f'Feature count: {len(FEATURES_FINAL_28)}')
print(f'Missing from normalized file: {missing or "none"}')
print(f'Leakage in features: {leakage_check or "none (clean)"}  ← must be empty')


In [ ]:
# ── v2d  CV comparison: v1-16 vs L1-58 vs deduped-28 ────────────────────
FEATURES_V1 = [
    'age_years', 'gender_female', 'med_count',
    'bpq020___ever_told_you_had_high_blood_pressure',
    'kiq480___how_many_times_urinate_in_night?',
    'paq650___vigorous_recreational_activities',
    'dpq040___feeling_tired_or_having_little_energy',
    'kiq026___ever_had_kidney_stones?',
    'kiq022___ever_told_you_had_weak/failing_kidneys?',
    'mcq160a___ever_told_you_had_arthritis',
    'kiq005___how_often_have_urinary_leakage?',
    'total_cholesterol_mg_dl', 'hdl_cholesterol_mg_dl',
    'triglycerides_mg_dl', 'uacr_mg_g', 'serum_creatinine_mg_dl',
]
FEATURES_V1 = [f for f in FEATURES_V1 if f in df.columns]

print('── 5-fold CV comparison ──────────────────────────────────────────────')
y_v1, p_v1, auc_v1 = run_cv(FEATURES_V1, f'v1-{len(FEATURES_V1)} (baseline)')
y_l1, p_l1, auc_l1 = run_cv(L1_SELECTED, f'L1-{len(L1_SELECTED)} (all survivors)')
y_28, p_28, auc_28 = run_cv(FEATURES_FINAL_28, f'deduped-{len(FEATURES_FINAL_28)} (final)')
print()
print(f'Delta vs v1: L1={auc_l1-auc_v1:+.4f}, deduped={auc_28-auc_v1:+.4f}')


In [ ]:
# ── v2e  OOF threshold sweep on deduped-28 ───────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

df_fin = df[FEATURES_FINAL_28 + [TARGET_COL]].dropna(subset=[TARGET_COL])
X_fin = df_fin[FEATURES_FINAL_28]
y_fin = df_fin[TARGET_COL].astype(int)

cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
prob_oof = cross_val_predict(build_pipe(1.0), X_fin, y_fin, cv=cv, method='predict_proba')[:,1]

print(f'AUC (5-fold OOF): {roc_auc_score(y_fin, prob_oof):.4f}')
print(f'Avg Precision:    {average_precision_score(y_fin, prob_oof):.4f}')
print(f'Prevalence:       {y_fin.mean():.3f} ({y_fin.sum()}/{len(y_fin)})')
print()

# Sweep thresholds: find recommended (max recall at precision >= 0.17)
thresholds = np.arange(0.05, 0.96, 0.05)
rows = []
for t in thresholds:
    pred = (prob_oof >= t).astype(int)
    prec = precision_score(y_fin, pred, zero_division=0)
    rec  = recall_score(y_fin, pred, zero_division=0)
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    rows.append({'thr':t, 'prec':prec, 'rec':rec, 'f1':f1, 'flagged':pred.mean()})
df_sw = pd.DataFrame(rows)

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>7} {'Flagged%':>10}")
for _, r in df_sw.iterrows():
    gate_m  = ' ← gate'    if abs(r['thr']-0.35)<0.01 else ''
    prec_m  = ' ← prec>=0.17' if r['prec'] >= 0.17 else ''
    print(f"{r['thr']:>10.2f} {r['prec']:>10.3f} {r['rec']:>8.3f} {r['f1']:>7.3f} {r['flagged']:>10.1%}{gate_m}{prec_m}")

# Recommended threshold: lowest where precision >= 0.17 (tolerance 0.03)
eligible = df_sw[df_sw['prec'] >= 0.17]
if len(eligible) > 0:
    best_row = eligible.iloc[0]   # lowest threshold = highest recall
    best_thr = best_row['thr']
    print(f'\nRecommended threshold: {best_thr:.2f}')
    print(f'  precision={best_row["prec"]:.3f}, recall={best_row["rec"]:.3f}, flagged={best_row["flagged"]:.1%}')
else:
    best_thr = 0.35  # fallback to gate
    print('No threshold achieves precision >= 0.17; using gate=0.35')


In [ ]:
# ── v2f  Final model coefficients ────────────────────────────────────────
pipe_coef = build_pipe(1.0)
pipe_coef.fit(X_fin, y_fin)

imp_step = pipe_coef.named_steps['imp']
miss_names_fin = [f'miss__{FEATURES_FINAL_28[i]}' for i in imp_step.indicator_.features_]
all_names_fin = FEATURES_FINAL_28 + miss_names_fin
coefs_fin = pipe_coef.named_steps['clf'].coef_[0]

coef_df = pd.DataFrame({'feature': all_names_fin, 'coef': coefs_fin})
coef_df = coef_df[~coef_df['feature'].str.startswith('miss__')]
coef_df = coef_df.sort_values('coef', key=abs, ascending=False)
print('Top coefficients (L2 final model, trained on all data):')
print(coef_df.to_string(index=False))


In [ ]:
# ── v2g  Save final model + metadata ─────────────────────────────────────
from datetime import datetime

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

# Retrain on all data
pipe_final = build_pipe(1.0)
pipe_final.fit(X_fin, y_fin)

fname  = 'electrolyte_imbalance_lr_deduped28_L2_v2.joblib'
mfname = fname.replace('.joblib', '_metadata.json')

joblib.dump(pipe_final, os.path.join(MODEL_OUT_DIR, fname))

metadata = {
    'model': fname, 'version': 'v2', 'condition': 'electrolyte_imbalance',
    'algorithm': 'LogisticRegression L2 C=1.0',
    'data_source': 'nhanes_merged_adults_final_normalized.csv',
    'n_train': int(len(df_fin)),
    'prevalence': round(float(y_fin.mean()), 4),
    'features': FEATURES_FINAL_28,
    'n_features': len(FEATURES_FINAL_28),
    'cv_folds': CV_FOLDS,
    'cv_auc_mean': round(float(roc_auc_score(y_fin, prob_oof)), 4),
    'cv_avg_precision': round(float(average_precision_score(y_fin, prob_oof)), 4),
    'pipeline_gate': 0.35,
    'pipeline_gate_rationale': 'Global routing gate: scores above 0.35 passed to next pipeline step',
    'recommended_threshold': float(best_thr),
    'recommended_threshold_criterion': 'Lowest threshold where precision >= 0.17, maximising recall',
    'pipeline_steps': [
        'SimpleImputer(strategy=median, add_indicator=True)',
        'LogisticRegression(L2, class_weight=balanced, C=1.0)',
    ],
    'leakage_removed': [
        'LBXSNASI_sodium_mmol_l (direct electrolyte lab)',
        'LBXSKSI_potassium_mmol_l (direct electrolyte lab)',
        'LBXSCA_total_calcium_mg_dl (direct electrolyte lab)',
        'sodium_mmol_l, potassium_mmol_l, chloride_mmol_l, bicarbonate_mmol_l, co2_mmol_l (target components)',
    ],
    'duplicates_removed': {
        'waist_cm': 'r=0.813 with weight_kg; weight_kg kept (stronger coef)',
        'sld013': 'r=0.451 with sld012; sld012 kept (stronger coef)',
        'rhq131': 'coef -0.054, below threshold',
        'kiq450': 'coef +0.052, below threshold',
    },
    'selection_rationale': (
        'L1 on all 79 roadmap features → 58 survivors → '
        'remove duplicates (r≥0.45) + weak coefs → 28 features. '
        'AUC identical to L1-58 (0.7166).'
    ),
    'changes_from_v1': [
        'Pre-normalized z-scored data', 'No StandardScaler',
        'SimpleImputer add_indicator=True', 'gender_female binary',
        '5-fold CV (OOF)', 'No dict wrapping', 'Global pipeline gate 0.35',
        'alcohol_any_risk_signal replaced by alq111/alq151/alq130',
        'Expanded to all 79 roadmap features then L1-selected and deduplicated',
    ],
    'created_at': datetime.utcnow().isoformat() + 'Z',
}

with open(os.path.join(MODEL_OUT_DIR, mfname), 'w') as f:
    _json.dump(metadata, f, indent=2)

print(f'Saved: {MODEL_OUT_DIR}/{fname}')
print(f'Saved: {MODEL_OUT_DIR}/{mfname}')
print(f'\nmodel_runner: "electrolyte_imbalance" -> "{fname}"')
print(f'\nSummary:')
print(f'  Features:            {len(FEATURES_FINAL_28)}')
print(f'  AUC (5-fold OOF):   {metadata["cv_auc_mean"]:.4f}  (v1 approx ~0.66-0.68)')
print(f'  Avg Precision:       {metadata["cv_avg_precision"]:.4f}')
print(f'  Prevalence:          {y_fin.mean():.3f}')
print(f'  Pipeline gate:       0.35  (R@gate={recall_score(y_fin,(prob_oof>=0.35).astype(int),zero_division=0):.3f})')
print(f'  Recommended thr:     {best_thr:.2f}  (P={best_row["prec"]:.3f}, R={best_row["rec"]:.3f})')


---
## v2 RF+cal — Comparison Attempt (kept LR)
RF+isotonic calibration tested across C=0.01–0.10 (15–50 features).

| Model | Feats | AUC | AUPRC | best t | prec | recall |
|---|---|---|---|---|---|---|
| **LR L2 (current)** | 28 | **0.7169** | **0.1797** | **0.60** | **0.175** | **0.459** |
| RF+cal C=0.10 | 50 | 0.6773 | 0.1506 | 0.15 | 0.197 | 0.202 |
| RF+cal C=0.03 | 30 | 0.6828 | 0.1553 | 0.15 | 0.193 | 0.237 |
| RF+cal C=0.01 | 15 | 0.6725 | 0.1474 | 0.15 | 0.200 | 0.263 |

**Decision: keep LR.** RF+cal AUC drops −0.034, AUPRC drops −0.024. RF collapses above t=0.50 (zero recall). LR at t=0.60 gives prec=17.5%, recall=45.9% — significantly more useful. Moderate prevalence (7.8%) with questionnaire features is well-suited to linear boundary.
